# Phase 8: Semantic family submission

This notebook reproduces the validated Phase 6 training pipeline, reconstructs the
32 hidden test families without using labels, averages probabilities within each
family, and optionally applies a fixed semantic intent router.

The router uses generic category definitions and text patterns. It never references
`ComplaintId`, exact test sentences, hidden labels, or desired test class counts.

The notebook only writes `/kaggle/working/submission.csv`. It does not submit to the
competition automatically.


## 1. Setup and frozen configuration

Attach these inputs to the Kaggle notebook before running:

- the ComplaintSense competition data;
- the `deberta-v3-base` Kaggle dataset used in the prior phases;
- `scraped_train_data.csv` used in the prior phases.


In [ ]:
import os
import gc
import glob
import json
import random
import re
import time
from pathlib import Path
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

import sklearn
import transformers
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedGroupKFold
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("Enable a GPU accelerator in the Kaggle notebook settings.")

print("device       :", DEVICE)
print("gpu          :", torch.cuda.get_device_name(0))
print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("scikit-learn:", sklearn.__version__)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Exact Kaggle mounts used by the executed Phase 5 notebook.
MODEL_PATH = "/kaggle/input/datasets/profzubby/deberta-v3-base"
SCRAPED_PATH = "/kaggle/input/datasets/profzubby/scraped-train-data/scraped_train_data.csv"

SEEDS = [42, 1337, 7]
N_SPLITS = 5
MAX_LEN = 192
BATCH_SIZE = 16
PRED_BATCH_SIZE = 32
NUM_WORKERS = 0

# External pretraining.
HEAD_WARMUP_EPOCHS = 1
HEAD_WARMUP_LR = 5e-4
STAGE1_EPOCHS = 3
STAGE1_ENCODER_LR = 1e-5
STAGE1_HEAD_LR = 5e-5

# Official-data adaptation. Epoch 4 is frozen from Phase 5c. The winning epoch-4
# snapshot came from a six-epoch scheduled run, so the scheduler horizon remains
# six even though this final notebook stops after saving epoch 4.
STAGE2_EPOCHS = 4
STAGE2_SCHEDULE_EPOCHS = 6
STAGE2_ENCODER_LR = 1e-5
STAGE2_HEAD_LR = 5e-5
CONSISTENCY_WEIGHT = 0.15

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0
LABEL_SMOOTHING = 0.05

# Preserve the tokenizer-loading behavior used in the executed Phase 5 run.
USE_FAST_TOKENIZER = False

# Use "family_only" for the conservative challenger or
# "semantic_guardrails" for the higher-upside submission.
SUBMISSION_MODE = "semantic_guardrails"
ALLOWED_SUBMISSION_MODES = {"family_only", "semantic_guardrails"}
if SUBMISSION_MODE not in ALLOWED_SUBMISSION_MODES:
    raise ValueError(f"Unknown SUBMISSION_MODE: {SUBMISSION_MODE}")

WORK_DIR = Path("/kaggle/working/complaintsense_phase8")
WORK_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
OOF_PATH = Path("/kaggle/working/phase8_oof_predictions.csv")
SCORES_PATH = Path("/kaggle/working/phase8_validation_scores.csv")
TEST_PROBS_PATH = Path("/kaggle/working/phase8_test_probabilities.npy")
FAMILY_AUDIT_PATH = Path("/kaggle/working/phase8_test_family_audit.csv")
RULE_AUDIT_PATH = Path("/kaggle/working/phase8_rule_audit.csv")

print(f"submission mode: {SUBMISSION_MODE}")
print(f"frozen run: {len(SEEDS)} seeds x {N_SPLITS} folds x {STAGE2_EPOCHS} target epochs")


## 2. Load and audit all data

Competition files are located recursively so the notebook remains robust to
Kaggle mount-name changes. If several copies are attached, the competition
mount is preferred.



In [ ]:
train_hits = glob.glob("/kaggle/input/**/train_complaints.csv", recursive=True)
test_hits = glob.glob("/kaggle/input/**/test_complaints.csv", recursive=True)
common_bases = sorted(
    set(map(os.path.dirname, train_hits)) & set(map(os.path.dirname, test_hits)),
    key=lambda path: ("competitions" not in path, path),
)
if not common_bases:
    raise FileNotFoundError("Attach the ComplaintSense competition data.")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model dataset not found: {MODEL_PATH}")
if not os.path.exists(SCRAPED_PATH):
    raise FileNotFoundError(f"External dataset not found: {SCRAPED_PATH}")

BASE = common_bases[0]
comp = pd.read_csv(os.path.join(BASE, "train_complaints.csv"))
test = pd.read_csv(os.path.join(BASE, "test_complaints.csv"))
scraped = pd.read_csv(SCRAPED_PATH)

TEXT, LABEL, GROUP = "text", "Category", "FamilyId"
SCR_TEXT, SCR_LABEL = "review_text", "category"

required_train = {"ComplaintId", TEXT, LABEL, GROUP}
required_test = {"ComplaintId", TEXT}
required_external = {SCR_TEXT, SCR_LABEL}
if not required_train.issubset(comp.columns):
    raise ValueError(f"Official train is missing: {required_train - set(comp.columns)}")
if not required_test.issubset(test.columns):
    raise ValueError(f"Official test is missing: {required_test - set(test.columns)}")
if not required_external.issubset(scraped.columns):
    raise ValueError(f"External data is missing: {required_external - set(scraped.columns)}")

CATS = np.array(sorted(comp[LABEL].astype(str).unique()))
VALID = set(CATS)
label2id = {category: i for i, category in enumerate(CATS)}
id2label = {i: category for category, i in label2id.items()}

EXPECTED_CATEGORIES = {
    "billing", "product_defect", "delivery_shipping", "refund_return",
    "customer_service", "account_access", "fraud_unauthorized",
    "warranty_repair", "subscription_cancel", "general_inquiry",
}
if VALID != EXPECTED_CATEGORIES:
    raise ValueError(f"Unexpected category set: {sorted(VALID)}")
if comp.groupby(GROUP)[LABEL].nunique().max() != 1:
    raise ValueError("A FamilyId spans multiple labels, so group validation is invalid.")


def canonicalize(text):
    """Remove only the synthetic wrappers identified in the official training data."""
    text = str(text).strip()
    text = re.sub(
        r"^(Writing to complain:|Flagging an issue:|Hello,|Hi support\s*[—-]|Case update:)\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"\s*Please advise\.?$", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"\bI'm reporting that I\b", "I", text, flags=re.IGNORECASE)
    text = re.sub(r"\bI'm reporting that\b", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bbilled\b", "charged", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()


comp["canonical_text"] = comp[TEXT].map(canonicalize)
test["canonical_text"] = test[TEXT].map(canonicalize)

# Normalize external labels and remove invalid or duplicate rows.
s = scraped.copy()
s[SCR_TEXT] = s[SCR_TEXT].astype(str).str.strip()
s[SCR_LABEL] = s[SCR_LABEL].astype(str).str.strip().str.lower()
external_before = len(s)
s = s[
    s[SCR_LABEL].isin(VALID)
    & s[SCR_TEXT].ne("")
    & s[SCR_TEXT].str.lower().ne("nan")
].drop_duplicates(subset=[SCR_TEXT, SCR_LABEL]).reset_index(drop=True)
s["canonical_text"] = s[SCR_TEXT].map(canonicalize)

# Leakage guard: no external item may exactly match an official train or test
# item after the fixed, label-free canonicalization above.
official_keys = set(comp["canonical_text"]) | set(test["canonical_text"])
overlap_mask = s["canonical_text"].isin(official_keys)
removed_overlap = int(overlap_mask.sum())
s = s.loc[~overlap_mask].reset_index(drop=True)
if s["canonical_text"].isin(official_keys).any():
    raise AssertionError("External-to-official exact-overlap removal failed.")

X_RAW = comp[TEXT].astype(str).values
X_CANON = comp["canonical_text"].astype(str).values
y = comp[LABEL].astype(str).values
g = comp[GROUP].values
TEST_RAW = test[TEXT].astype(str).values
TEST_CANON = test["canonical_text"].astype(str).values
eX = s[SCR_TEXT].astype(str).values
ey = s[SCR_LABEL].astype(str).values

print("competition base       :", BASE)
print(f"official train         : {len(comp)} rows, {comp[GROUP].nunique()} families")
print(f"official test          : {len(test)} rows")
print(f"external input         : {external_before:,} rows")
print(f"external retained      : {len(s):,} rows")
print(f"exact overlaps removed : {removed_overlap}")
print("categories             :", list(CATS))

# This reconstructs official training families exactly, but not all test families.
# Therefore the final prediction path is intentionally row-level.
train_grouping_exact = (
    comp["canonical_text"].nunique() == comp[GROUP].nunique()
    and comp.groupby("canonical_text")[GROUP].nunique().max() == 1
    and comp.groupby("canonical_text").size().eq(5).all()
)
test_group_sizes = test.groupby("canonical_text").size()
test_grouping_exact = (
    test["canonical_text"].nunique() == 32
    and test_group_sizes.eq(5).all()
)
print("train family reconstruction exact:", train_grouping_exact)
print("test family reconstruction exact :", test_grouping_exact)
if not test_grouping_exact:
    print("Row-level test inference retained. Recovered group-size counts:")
    print(test_group_sizes.value_counts().sort_index().to_string())



## 3. Tokenization, datasets, and model helpers

FP32 is used throughout. Phase 4 showed a collapsed AMP seed, so AMP is
deliberately absent from this pipeline.



In [ ]:
try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_PATH, use_fast=USE_FAST_TOKENIZER
    )
except Exception as tokenizer_error:
    raise RuntimeError(
        "The tokenizer could not be loaded with the frozen Phase 5 settings. "
        "Do not silently switch tokenizer variants in a final reproducibility run."
    ) from tokenizer_error

print("tokenizer:", type(tokenizer).__name__, "| fast:", tokenizer.is_fast)
if "deberta" not in type(tokenizer).__name__.lower():
    raise RuntimeError("The tokenizer does not match the DeBERTa checkpoint.")


class SingleDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = None if labels is None else [label2id[str(v)] for v in labels]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        label = -1 if self.labels is None else self.labels[i]
        return self.texts[i], label


def collate_single(batch):
    texts, labels = zip(*batch)
    encoded = tokenizer(
        list(texts),
        truncation=True,
        max_length=MAX_LEN,
        padding=True,
        return_tensors="pt",
    )
    encoded["labels"] = torch.tensor(labels, dtype=torch.long)
    return encoded


class DualViewDataset(Dataset):
    def __init__(self, raw_texts, canonical_texts, labels):
        self.raw = list(raw_texts)
        self.canonical = list(canonical_texts)
        self.labels = [label2id[str(v)] for v in labels]

    def __len__(self):
        return len(self.raw)

    def __getitem__(self, i):
        return self.raw[i], self.canonical[i], self.labels[i]


def collate_dual(batch):
    raw, canonical, labels = zip(*batch)
    raw_encoded = tokenizer(
        list(raw), truncation=True, max_length=MAX_LEN,
        padding=True, return_tensors="pt"
    )
    canonical_encoded = tokenizer(
        list(canonical), truncation=True, max_length=MAX_LEN,
        padding=True, return_tensors="pt"
    )
    return raw_encoded, canonical_encoded, torch.tensor(labels, dtype=torch.long)


def make_single_loader(texts, labels=None, batch_size=BATCH_SIZE, shuffle=False):
    return DataLoader(
        SingleDataset(texts, labels),
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate_single,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )


def make_dual_loader(raw, canonical, labels, shuffle=True):
    return DataLoader(
        DualViewDataset(raw, canonical, labels),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_dual,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )


def balanced_weights(labels):
    counts = pd.Series(labels).value_counts()
    values = [
        len(labels) / (len(CATS) * max(int(counts.get(category, 0)), 1))
        for category in CATS
    ]
    return torch.tensor(values, dtype=torch.float32, device=DEVICE)


def split_parameters(model):
    base_ids = {id(parameter) for parameter in model.base_model.parameters()}
    encoder = [parameter for parameter in model.parameters() if id(parameter) in base_ids]
    head = [parameter for parameter in model.parameters() if id(parameter) not in base_ids]
    return encoder, head


def set_encoder_trainable(model, trainable):
    for parameter in model.base_model.parameters():
        parameter.requires_grad = trainable


def differential_optimizer(model, encoder_lr, head_lr):
    encoder, head = split_parameters(model)
    return torch.optim.AdamW(
        [
            {"params": encoder, "lr": encoder_lr},
            {"params": head, "lr": head_lr},
        ],
        weight_decay=WEIGHT_DECAY,
    )


@torch.no_grad()
def predict_multiclass(model, texts):
    model.eval()
    chunks = []
    loader = make_single_loader(
        texts, labels=None, batch_size=PRED_BATCH_SIZE, shuffle=False
    )
    for batch in loader:
        batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}
        batch.pop("labels")
        logits = model(**batch).logits
        chunks.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.vstack(chunks)


def predict_dual(model, raw, canonical):
    return 0.5 * predict_multiclass(model, raw) + 0.5 * predict_multiclass(model, canonical)


def macro_f1(probs, truth=y):
    predicted = CATS[np.asarray(probs).argmax(axis=1)]
    return f1_score(truth, predicted, average="macro", zero_division=0)


def release_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


print("model helpers ready")



## 4. Stage 1: stable external pretraining

Three seeds are trained independently. Encoder warm-up, class weights, label
smoothing, gradient clipping, and FP32 match the successful Phase 5c experiment.
A seed-health gate uses external training loss only. Official labels never
determine whether a seed is accepted.



In [ ]:
def train_external(seed, save_dir):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH,
        num_labels=len(CATS),
        torch_dtype=torch.float32,
    ).to(DEVICE)
    loader = make_single_loader(eX, ey, batch_size=BATCH_SIZE, shuffle=True)
    loss_fn = nn.CrossEntropyLoss(
        weight=balanced_weights(ey),
        label_smoothing=LABEL_SMOOTHING,
    )

    # Warm only the newly initialized pooler and classification head.
    set_encoder_trainable(model, False)
    _, head = split_parameters(model)
    warm_optimizer = torch.optim.AdamW(
        head, lr=HEAD_WARMUP_LR, weight_decay=WEIGHT_DECAY
    )
    for epoch in range(HEAD_WARMUP_EPOCHS):
        model.train()
        running = 0.0
        progress = tqdm(loader, desc=f"head-warmup-s{seed}", leave=False)
        for step, batch in enumerate(progress, 1):
            batch = {
                key: value.to(DEVICE, non_blocking=True)
                for key, value in batch.items()
            }
            labels_tensor = batch.pop("labels")
            warm_optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(**batch).logits, labels_tensor)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(head, MAX_GRAD_NORM)
            warm_optimizer.step()
            running += loss.item()
            progress.set_postfix(loss=f"{running / step:.4f}")
        print(f"seed {seed} head warm-up loss={running / len(loader):.4f}")

    set_encoder_trainable(model, True)
    optimizer = differential_optimizer(model, STAGE1_ENCODER_LR, STAGE1_HEAD_LR)
    total_steps = max(len(loader) * STAGE1_EPOCHS, 1)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(WARMUP_RATIO * total_steps), total_steps
    )
    history = []
    for epoch in range(1, STAGE1_EPOCHS + 1):
        model.train()
        running = 0.0
        progress = tqdm(
            loader,
            desc=f"external-s{seed}-{epoch}/{STAGE1_EPOCHS}",
            leave=False,
        )
        for step, batch in enumerate(progress, 1):
            batch = {
                key: value.to(DEVICE, non_blocking=True)
                for key, value in batch.items()
            }
            labels_tensor = batch.pop("labels")
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(**batch).logits, labels_tensor)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            running += loss.item()
            progress.set_postfix(loss=f"{running / step:.4f}")
        epoch_loss = running / len(loader)
        history.append(epoch_loss)
        print(f"seed {seed} external epoch {epoch}: loss={epoch_loss:.4f}")

    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_dir, safe_serialization=True)
    tokenizer.save_pretrained(save_dir)
    return model, history


stage1_paths = {}
stage1_losses = {}
external_oof_by_seed = {}
external_test_by_seed = {}
stage1_log = []

for seed in SEEDS:
    started = time.time()
    checkpoint_path = WORK_DIR / f"external_seed_{seed}"
    print(f"\n===== EXTERNAL SEED {seed} =====")
    model, history = train_external(seed, checkpoint_path)
    stage1_paths[seed] = str(checkpoint_path)
    stage1_losses[seed] = history[-1]
    external_oof_by_seed[seed] = predict_dual(model, X_RAW, X_CANON)
    external_test_by_seed[seed] = predict_dual(model, TEST_RAW, TEST_CANON)
    stage1_log.append(
        {
            "seed": seed,
            "final_external_training_loss": history[-1],
            "diagnostic_official_macro_f1": macro_f1(external_oof_by_seed[seed]),
            "minutes": (time.time() - started) / 60,
        }
    )
    release_model(model)

loss_values = np.array(list(stage1_losses.values()))
median_loss = float(np.median(loss_values))
loss_gate = max(1.35 * median_loss, median_loss + 0.25)
HEALTHY_SEEDS = [seed for seed in SEEDS if stage1_losses[seed] <= loss_gate]
if len(HEALTHY_SEEDS) < 2:
    HEALTHY_SEEDS = sorted(SEEDS, key=stage1_losses.get)[:2]
    print("WARNING: fewer than two seeds passed; using the two lowest-loss seeds.")

EXTERNAL_OOF = np.mean([external_oof_by_seed[seed] for seed in HEALTHY_SEEDS], axis=0)
EXTERNAL_TEST = np.mean([external_test_by_seed[seed] for seed in HEALTHY_SEEDS], axis=0)

print("\nStage 1 seed audit")
display(pd.DataFrame(stage1_log).sort_values("final_external_training_loss").round(4))
print("loss gate    :", round(loss_gate, 4))
print("healthy seeds:", HEALTHY_SEEDS)
print(f"external-only diagnostic macro F1: {macro_f1(EXTERNAL_OOF):.4f}")



## 5. Stage 2: family-safe target adaptation

Each fold holds out complete `FamilyId` groups. Raw and canonical versions of
every training complaint share a label, and a symmetric consistency term makes
predictions less sensitive to the synthetic wrappers.

The same five fold assignments are reused for all seeds. Test probabilities are
averaged over every healthy seed and fold. This is the exact cross-validation
ensemble design selected in Phase 5c.



In [ ]:
def train_target_fold(
    init_path,
    train_raw,
    train_canonical,
    train_labels,
    val_raw,
    val_canonical,
    test_raw,
    test_canonical,
    seed,
    fold,
):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        init_path,
        num_labels=len(CATS),
        torch_dtype=torch.float32,
    ).to(DEVICE)
    loader = make_dual_loader(
        train_raw, train_canonical, train_labels, shuffle=True
    )
    loss_fn = nn.CrossEntropyLoss(
        weight=balanced_weights(train_labels),
        label_smoothing=LABEL_SMOOTHING,
    )
    optimizer = differential_optimizer(model, STAGE2_ENCODER_LR, STAGE2_HEAD_LR)
    total_steps = max(len(loader) * STAGE2_SCHEDULE_EPOCHS, 1)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(WARMUP_RATIO * total_steps), total_steps
    )

    history = []
    for epoch in range(1, STAGE2_EPOCHS + 1):
        model.train()
        running = 0.0
        progress = tqdm(
            loader,
            desc=f"target-f{fold}-s{seed}-{epoch}/{STAGE2_EPOCHS}",
            leave=False,
        )
        for step, (raw_batch, canonical_batch, labels_tensor) in enumerate(progress, 1):
            raw_batch = {
                key: value.to(DEVICE, non_blocking=True)
                for key, value in raw_batch.items()
            }
            canonical_batch = {
                key: value.to(DEVICE, non_blocking=True)
                for key, value in canonical_batch.items()
            }
            labels_tensor = labels_tensor.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            raw_logits = model(**raw_batch).logits
            canonical_logits = model(**canonical_batch).logits
            supervised_loss = 0.5 * (
                loss_fn(raw_logits, labels_tensor)
                + loss_fn(canonical_logits, labels_tensor)
            )
            raw_logp = F.log_softmax(raw_logits, dim=-1)
            canonical_logp = F.log_softmax(canonical_logits, dim=-1)
            raw_prob = raw_logp.exp().detach()
            canonical_prob = canonical_logp.exp().detach()
            consistency_loss = 0.5 * (
                F.kl_div(raw_logp, canonical_prob, reduction="batchmean")
                + F.kl_div(canonical_logp, raw_prob, reduction="batchmean")
            )
            loss = supervised_loss + CONSISTENCY_WEIGHT * consistency_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            running += loss.item()
            progress.set_postfix(loss=f"{running / step:.4f}")

        epoch_loss = running / len(loader)
        history.append(epoch_loss)
        print(f"fold {fold}, seed {seed}, epoch {epoch}: loss={epoch_loss:.4f}")

        # Phase 5 also predicted at epoch 2 before continuing. Repeating that
        # evaluation preserves the deterministic RNG sequence used by the
        # validated epoch-4 model. The probabilities are intentionally discarded.
        if epoch == 2:
            _ = predict_dual(model, val_raw, val_canonical)
            _ = predict_dual(model, test_raw, test_canonical)

    val_probs = predict_dual(model, val_raw, val_canonical)
    test_probs = predict_dual(model, test_raw, test_canonical)
    release_model(model)
    return val_probs, test_probs, history


splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=42
)
fold_splits = list(splitter.split(X_RAW, y, g))

# Explicit leakage checks for every fold.
fold_id = np.full(len(y), -1, dtype=int)
for fold_zero, (train_idx, val_idx) in enumerate(fold_splits):
    train_families = set(g[train_idx])
    val_families = set(g[val_idx])
    if train_families & val_families:
        raise AssertionError(f"Family leakage in fold {fold_zero + 1}.")
    fold_id[val_idx] = fold_zero
if (fold_id < 0).any():
    raise AssertionError("Some official rows did not receive an OOF fold.")

SEQ_OOF_BY_SEED = np.zeros(
    (len(HEALTHY_SEEDS), len(comp), len(CATS)), dtype=np.float32
)
SEQ_TEST_BY_SEED_FOLD = np.zeros(
    (len(HEALTHY_SEEDS), N_SPLITS, len(test), len(CATS)), dtype=np.float32
)
stage2_log = []

for fold_zero, (train_idx, val_idx) in enumerate(fold_splits):
    fold = fold_zero + 1
    print(f"\n{'=' * 70}\nTARGET FOLD {fold}/{N_SPLITS}\n{'=' * 70}")
    for seed_position, seed in enumerate(HEALTHY_SEEDS):
        started = time.time()
        val_probs, test_probs, history = train_target_fold(
            stage1_paths[seed],
            X_RAW[train_idx],
            X_CANON[train_idx],
            y[train_idx],
            X_RAW[val_idx],
            X_CANON[val_idx],
            TEST_RAW,
            TEST_CANON,
            seed,
            fold,
        )
        SEQ_OOF_BY_SEED[seed_position, val_idx] = val_probs
        SEQ_TEST_BY_SEED_FOLD[seed_position, fold_zero] = test_probs
        stage2_log.append(
            {
                "fold": fold,
                "seed": seed,
                "epoch": STAGE2_EPOCHS,
                "fold_macro_f1": macro_f1(val_probs, y[val_idx]),
                "final_training_loss": history[-1],
                "minutes": (time.time() - started) / 60,
            }
        )

SEQ_OOF = SEQ_OOF_BY_SEED.mean(axis=0)
SEQ_TEST = SEQ_TEST_BY_SEED_FOLD.mean(axis=(0, 1))

print("\nTarget-adaptation run audit")
display(pd.DataFrame(stage2_log).round(4))
print(f"family-safe sequential OOF macro F1: {macro_f1(SEQ_OOF):.4f}")



## 6. Family reconstruction and semantic intent guardrails

The official data description states that test contains 32 five-row semantic
families. The fixed canonicalization recovers 31 families exactly, plus one group
of four and one singleton. The code below merges that residual pair only when its
string similarity is high, then requires exactly 32 groups of five.

The semantic router is fixed before generating predictions. Rules express broad
intent distinctions such as a neutral policy question versus a complaint, an
access failure versus observed account fraud, and a repair problem versus an
initial product defect.

The semantic-rule validation score is a development estimate because the rules
were selected after error analysis. The base family score remains the cleanest
generalization estimate.


In [ ]:
def aggregate_probs(probs, groups):
    """Broadcast each family's mean probability vector back to its rows."""
    probs = np.asarray(probs)
    groups = np.asarray(groups)
    result = np.zeros_like(probs)

    for group_value in pd.unique(groups):
        mask = groups == group_value
        result[mask] = probs[mask].mean(axis=0, keepdims=True)

    return result

In [ ]:
def account_fraud_rule(sequential_probs, external_probs):
    """Frozen Phase 6 rule, retained unchanged for reproducibility."""
    result = sequential_probs.copy()
    sequential_pred = CATS[sequential_probs.argmax(axis=1)]
    external_pred = CATS[external_probs.argmax(axis=1)]
    mask = (
        (external_pred == "account_access")
        & (sequential_pred == "fraud_unauthorized")
    )
    account_id = label2id["account_access"]
    result[mask] = 0.0
    result[mask, account_id] = 1.0
    return result, mask


def recover_test_families(canonical_texts):
    """Recover 32 groups of five using labels-free text structure only."""
    keys = np.asarray(canonical_texts, dtype=object).copy()
    counts = pd.Series(keys).value_counts()

    if len(counts) == 32 and counts.eq(5).all():
        return keys, {"repair_used": False, "similarity": None}

    expected_size_profile = {1: 1, 4: 1, 5: 31}
    observed_size_profile = counts.value_counts().sort_index().to_dict()
    if observed_size_profile != expected_size_profile:
        raise ValueError(
            "Unexpected test family structure after canonicalization: "
            f"{observed_size_profile}"
        )

    singleton_key = counts[counts.eq(1)].index[0]
    four_row_key = counts[counts.eq(4)].index[0]
    similarity = SequenceMatcher(
        None, str(singleton_key).lower(), str(four_row_key).lower()
    ).ratio()
    if similarity < 0.80:
        raise ValueError(
            "The unmatched test rows are not similar enough for safe family repair: "
            f"similarity={similarity:.3f}"
        )

    keys[keys == singleton_key] = four_row_key
    repaired_counts = pd.Series(keys).value_counts()
    if len(repaired_counts) != 32 or not repaired_counts.eq(5).all():
        raise AssertionError("Test-family repair did not produce 32 groups of five.")

    return keys, {
        "repair_used": True,
        "similarity": float(similarity),
        "singleton_key": str(singleton_key),
        "matched_group_key": str(four_row_key),
    }


def semantic_rule_flags(text):
    """Generic rules derived from the published ten-category taxonomy."""
    text = str(text)
    question_like = bool(
        re.match(
            r"^(what|when|where|why|who|which|how|does|do|is|are|can|could|would|will|should)\b",
            text,
            flags=re.IGNORECASE,
        )
        or "?" in text
    )
    adverse_event = bool(
        re.search(
            r"\b(delayed|late|lost|missing|wrong|broken|failed|fails|failing|still|never|"
            r"refused|denied|charged|charges|damaged|stuck)\b",
            text,
            flags=re.IGNORECASE,
        )
    )
    neutral_question = question_like and not adverse_event

    observed_fraud = bool(
        re.search(
            r"\b(logins? from|login from|another country|cities I never visited|"
            r"unauthorized|unknown party)\b",
            text,
            flags=re.IGNORECASE,
        )
    )
    access_failure = bool(
        re.search(
            r"\b(locked out|account locked|cannot log in|can.t log in|login fails|"
            r"log in fails|sign[ -]?on loop|two[ -]?factor|2fa|codes? never arrive|"
            r"password reset.*(?:fail|expired)|prevents access|security lock)\b",
            text,
            flags=re.IGNORECASE,
        )
    ) and not observed_fraud

    explicit_fraud = bool(
        re.search(
            r"\b(unauthorized|unknown party|did not authorize|never activated|not mine|"
            r"unrecognized|micro-charges?|logins? from|login from another country)\b",
            text,
            flags=re.IGNORECASE,
        )
    )
    service_failure = bool(
        re.search(
            r"\b(conflicting answers|transferred me|callback.*never|"
            r"hold time.*(?:ended|without)|agent ended chat)\b",
            text,
            flags=re.IGNORECASE,
        )
    )
    warranty_or_repair = bool(
        re.search(
            r"\b(warranty|protection plan|repair(?:ed| center| appointment)?)\b",
            text,
            flags=re.IGNORECASE,
        )
    ) and not bool(
        re.search(
            r"\b(representative|conflicting answers)\b",
            text,
            flags=re.IGNORECASE,
        )
    )
    return_or_refund = bool(
        re.search(
            r"\b(return(?:ed|ing)?|refund|reimbursement|restocking fee)\b",
            text,
            flags=re.IGNORECASE,
        )
    )
    subscription_flow = bool(
        re.search(
            r"\b(cancel button|cancellation|trial converted|renewal|subscription|"
            r"membership|downgrade)\b",
            text,
            flags=re.IGNORECASE,
        )
    ) and not bool(
        re.search(
            r"\b(unknown|unauthorized|never activated|not mine|did not authorize)\b",
            text,
            flags=re.IGNORECASE,
        )
    )

    return {
        "neutral_question": neutral_question,
        "access_failure": access_failure,
        "explicit_fraud": explicit_fraud,
        "service_failure": service_failure,
        "warranty_or_repair": warranty_or_repair,
        "return_or_refund": return_or_refund,
        "subscription_flow": subscription_flow,
    }


RULE_SPECS = [
    ("neutral_question", "general_inquiry", None),
    (
        "access_failure",
        "account_access",
        {"fraud_unauthorized", "subscription_cancel", "customer_service", "billing"},
    ),
    (
        "explicit_fraud",
        "fraud_unauthorized",
        {"account_access", "billing", "subscription_cancel"},
    ),
    (
        "service_failure",
        "customer_service",
        {"warranty_repair", "delivery_shipping"},
    ),
    (
        "warranty_or_repair",
        "warranty_repair",
        {
            "product_defect", "subscription_cancel", "customer_service", "billing",
            "refund_return", "delivery_shipping",
        },
    ),
    ("return_or_refund", "refund_return", {"delivery_shipping", "billing"}),
    (
        "subscription_flow",
        "subscription_cancel",
        {"billing", "fraud_unauthorized"},
    ),
]


def apply_semantic_guardrails(probs, canonical_texts):
    """Apply at most one rule per row in a fixed, documented priority order."""
    result = np.asarray(probs).copy()
    texts = np.asarray(canonical_texts, dtype=object)
    flag_rows = [semantic_rule_flags(text) for text in texts]
    already_routed = np.zeros(len(result), dtype=bool)
    audit_rows = []

    for rule_name, target_label, allowed_predictions in RULE_SPECS:
        current_prediction = CATS[result.argmax(axis=1)]
        mask = np.array([row[rule_name] for row in flag_rows], dtype=bool)
        mask &= ~already_routed
        mask &= current_prediction != target_label
        if allowed_predictions is not None:
            mask &= np.isin(current_prediction, list(allowed_predictions))

        target_id = label2id[target_label]
        for row_index in np.flatnonzero(mask):
            audit_rows.append(
                {
                    "row_index": int(row_index),
                    "rule": rule_name,
                    "before": str(current_prediction[row_index]),
                    "after": target_label,
                    "canonical_text": str(texts[row_index]),
                }
            )
        result[mask] = 0.0
        result[mask, target_id] = 1.0
        already_routed |= mask

    return result, pd.DataFrame(audit_rows)


test_family_key, family_repair = recover_test_families(TEST_CANON)
print("test-family reconstruction:", family_repair)
print(pd.Series(test_family_key).value_counts().value_counts().sort_index().to_string())

# Preserve the inherited Phase 6 rule before family aggregation.
ROUTED_OOF, account_rule_oof_mask = account_fraud_rule(SEQ_OOF, EXTERNAL_OOF)
ROUTED_TEST, account_rule_test_mask = account_fraud_rule(SEQ_TEST, EXTERNAL_TEST)

# The label-free family prior is applied to both OOF and test probabilities.
FAMILY_BASE_OOF = aggregate_probs(SEQ_OOF, g)
FAMILY_ROUTED_OOF = aggregate_probs(ROUTED_OOF, g)
FAMILY_BASE_TEST = aggregate_probs(SEQ_TEST, test_family_key)
FAMILY_ROUTED_TEST = aggregate_probs(ROUTED_TEST, test_family_key)

SEMANTIC_OOF, semantic_oof_audit = apply_semantic_guardrails(
    FAMILY_BASE_OOF, X_CANON
)
FULL_OOF, full_oof_audit = apply_semantic_guardrails(
    FAMILY_ROUTED_OOF, X_CANON
)
SEMANTIC_TEST, semantic_test_audit = apply_semantic_guardrails(
    FAMILY_BASE_TEST, TEST_CANON
)
FULL_TEST, full_test_audit = apply_semantic_guardrails(
    FAMILY_ROUTED_TEST, TEST_CANON
)

score_table = pd.DataFrame(
    [
        {
            "method": "Phase 6 sequential model, row-level",
            "validation_macro_f1": macro_f1(SEQ_OOF),
            "estimate_type": "clean family-safe OOF",
        },
        {
            "method": "Phase 6 sequential model, family aggregation",
            "validation_macro_f1": macro_f1(FAMILY_BASE_OOF),
            "estimate_type": "clean family-safe OOF",
        },
        {
            "method": "family aggregation plus inherited account/fraud rule",
            "validation_macro_f1": macro_f1(FAMILY_ROUTED_OOF),
            "estimate_type": "development OOF; inherited Phase 6 rule",
        },
        {
            "method": "family aggregation plus semantic guardrails",
            "validation_macro_f1": macro_f1(SEMANTIC_OOF),
            "estimate_type": "development OOF; rules selected after error analysis",
        },
        {
            "method": "family aggregation plus inherited rule and semantic guardrails",
            "validation_macro_f1": macro_f1(FULL_OOF),
            "estimate_type": "development OOF; rules selected after error analysis",
        },
    ]
)
display(score_table.assign(validation_macro_f1=score_table["validation_macro_f1"].round(4)))
print(
    f"inherited account/fraud changes: {account_rule_oof_mask.sum()} OOF rows, "
    f"{account_rule_test_mask.sum()} test rows"
)
print(
    "semantic families changed on test:",
    0
    if full_test_audit.empty
    else full_test_audit["row_index"].map(dict(enumerate(test_family_key))).nunique(),
)

if SUBMISSION_MODE == "family_only":
    FINAL_OOF = FAMILY_ROUTED_OOF
    FINAL_TEST = FAMILY_ROUTED_TEST
else:
    FINAL_OOF = FULL_OOF
    FINAL_TEST = FULL_TEST


## 7. Generate, audit, and save `submission.csv`

Kaggle does not submit this file automatically. After a successful **Save & Run
All**, inspect the output and submit it manually from the competition interface.


In [ ]:
TEST_PRED = CATS[FINAL_TEST.argmax(axis=1)]
submission = pd.DataFrame(
    {
        "ComplaintId": test["ComplaintId"].values,
        "Category": TEST_PRED,
    }
)

family_prediction_counts = submission.assign(
    inferred_family=test_family_key
).groupby("inferred_family")["Category"].nunique()

checks = {
    "exact columns": list(submission.columns) == ["ComplaintId", "Category"],
    "160 rows": len(submission) == 160,
    "test row order preserved": np.array_equal(
        submission["ComplaintId"].values, test["ComplaintId"].values
    ),
    "valid category labels": set(submission["Category"]) <= VALID,
    "no missing values": bool(submission.notna().all().all()),
    "unique ComplaintId": submission["ComplaintId"].is_unique,
    "32 inferred test families": pd.Series(test_family_key).nunique() == 32,
    "five rows per inferred family": pd.Series(test_family_key).value_counts().eq(5).all(),
    "one prediction per inferred family": family_prediction_counts.eq(1).all(),
}

print("Submission checks")
for check, passed in checks.items():
    print(f"  {'PASS' if passed else 'FAIL'}: {check}")
if not all(checks.values()):
    raise ValueError("Submission validation failed. File was not written.")

submission.to_csv(SUBMISSION_PATH, index=False)
np.save(TEST_PROBS_PATH, FINAL_TEST)
score_table.to_csv(SCORES_PATH, index=False)

oof_output = comp[["ComplaintId", GROUP, LABEL, TEXT]].copy()
oof_output["fold"] = fold_id + 1
oof_output["phase6_row_prediction"] = CATS[SEQ_OOF.argmax(axis=1)]
oof_output["family_prediction"] = CATS[FAMILY_BASE_OOF.argmax(axis=1)]
oof_output["semantic_prediction"] = CATS[FULL_OOF.argmax(axis=1)]
oof_output.to_csv(OOF_PATH, index=False)

family_audit = test[["ComplaintId", TEXT]].copy()
family_audit["inferred_family"] = test_family_key
family_audit["phase6_row_prediction"] = CATS[SEQ_TEST.argmax(axis=1)]
family_audit["family_prediction"] = CATS[FAMILY_ROUTED_TEST.argmax(axis=1)]
family_audit["final_prediction"] = TEST_PRED
family_audit.to_csv(FAMILY_AUDIT_PATH, index=False)

rule_audit = full_test_audit.copy()
if rule_audit.empty:
    rule_audit = pd.DataFrame(
        columns=["row_index", "rule", "before", "after", "canonical_text"]
    )
rule_audit["ComplaintId"] = (
    rule_audit["row_index"].map(test["ComplaintId"]) if len(rule_audit) else pd.Series(dtype=int)
)
rule_audit["inferred_family"] = (
    rule_audit["row_index"].map(dict(enumerate(test_family_key)))
    if len(rule_audit)
    else pd.Series(dtype=object)
)
rule_audit.to_csv(RULE_AUDIT_PATH, index=False)

print(f"\nCreated: {SUBMISSION_PATH}")
print(f"Mode   : {SUBMISSION_MODE}")
print(f"Rows   : {len(submission)}")
print("\nPrediction counts")
print(submission["Category"].value_counts().reindex(CATS, fill_value=0).to_string())
print("\nRule changes by inferred family")
if rule_audit.empty:
    print("None")
else:
    display(
        rule_audit.drop_duplicates(["inferred_family", "rule"])[
            ["rule", "before", "after", "canonical_text"]
        ].reset_index(drop=True)
    )
display(submission.head(10))


## 8. Final reproducibility record

The manifest records the selected mode, validation estimates, family-recovery
decision, and all files generated by this run.


In [ ]:
run_manifest = {
    "method": "Phase 8 family aggregation with optional semantic guardrails",
    "submission_mode": SUBMISSION_MODE,
    "model_path": MODEL_PATH,
    "scraped_path": SCRAPED_PATH,
    "competition_base": BASE,
    "categories_sorted": list(CATS),
    "candidate_seeds": SEEDS,
    "healthy_seeds": HEALTHY_SEEDS,
    "folds": N_SPLITS,
    "stage1_epochs": STAGE1_EPOCHS,
    "stage2_epochs": STAGE2_EPOCHS,
    "stage2_scheduler_horizon_epochs": STAGE2_SCHEDULE_EPOCHS,
    "phase6_row_oof_macro_f1": float(macro_f1(SEQ_OOF)),
    "clean_family_oof_macro_f1": float(macro_f1(FAMILY_BASE_OOF)),
    "selected_development_oof_macro_f1": float(macro_f1(FINAL_OOF)),
    "semantic_score_is_development_estimate": SUBMISSION_MODE == "semantic_guardrails",
    "oof_rows_changed_by_inherited_rule": int(account_rule_oof_mask.sum()),
    "test_rows_changed_by_inherited_rule": int(account_rule_test_mask.sum()),
    "test_family_recovery": family_repair,
    "test_families": int(pd.Series(test_family_key).nunique()),
    "semantic_test_families_changed": int(
        0
        if full_test_audit.empty
        else full_test_audit["row_index"].map(dict(enumerate(test_family_key))).nunique()
    ),
    "external_exact_overlaps_removed": removed_overlap,
    "submission_path": str(SUBMISSION_PATH),
    "automatic_submission_called": False,
    "outputs": [
        str(SUBMISSION_PATH), str(SCORES_PATH), str(OOF_PATH),
        str(TEST_PROBS_PATH), str(FAMILY_AUDIT_PATH), str(RULE_AUDIT_PATH),
    ],
}
with open(WORK_DIR / "run_manifest.json", "w") as file:
    json.dump(run_manifest, file, indent=2)

print(json.dumps(run_manifest, indent=2))
print("\nDONE. submission.csv was generated. Nothing was submitted automatically.")
